<h1 style="margin-bottom: 5px;">Logo Classification</h1>
<h5 style="margin-top: 0; margin-bottom: 10px">
Artificial Neural Networks and Deep Learning D - Spring 2026<br>
Arman Kassam, Justin Kong, Shadab Sharif, and Tevin Park
</h5>
Logos are powerful visual signals that shape brand perception, influencing recognition, trust, and consumer behavior. In this project, we leverage deep learning to analyze and classify brand logos, uncovering patterns that link visual design to industry identity. By combining computer vision with structured labeling, we aim to explore how effectively a model can infer a company’s sector purely from its logo.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import os

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

ModuleNotFoundError: No module named 'cv2'

In [ ]:
CHECKPOINT_FILEPATH = '/kaggle/working/chkpts/checkpoint.model.keras'
TUNED_CHECKPOINT_FILEPATH = '/kaggle/working/chkpts/tuned-checkpoint.model.keras'

In [ ]:
strategy = tf.distribute.MirroredStrategy()

# dataset augmentation
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.1),
])

with strategy.scope():
    model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # freeze weights
    model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))

    x = data_augmentation(inputs)
    x = model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(10, activation='softmax')(x)

    model = keras.Model(inputs, outputs)

IndentationError: unexpected indent (3929643403.py, line 11)

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │        29,463 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,079,034 (15.56 MB)

 Trainable params: 29,463 (115.09 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
labeled = pd.read_csv("/kaggle/input/datasets/jkong05/c-logo-labels/labels_10cls.csv")
labeled["sector"] = labeled["sector"].astype("category")
labeled["label_idx"] = labeled["sector"].cat.codes

classes = labeled["sector"].cat.categories
num_classes = len(classes)

train_df, temp_df = train_test_split(labeled, test_size=0.2, stratify=labeled["sector"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["sector"], random_state=42)

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    
    # do not need to /= 255 because EfficientNetB0 already has a layer that rescales the input
    return tf.cast(img, tf.float32), label

train_data = tf.data.Dataset.from_tensor_slices((train_df["filepath"].values, train_df["label_idx"].values))
train_data = train_data.shuffle(1000).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

val_data = tf.data.Dataset.from_tensor_slices((val_df["filepath"].values, val_df["label_idx"].values))
val_data = val_data.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

test_data = tf.data.Dataset.from_tensor_slices((test_df["filepath"].values, test_df["label_idx"].values))
test_data = test_data.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

In [ ]:
with strategy.scope():
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', 
                    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_SparseTopKCategoricalAccuracy'),
        ]
    )

callbacks = [
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_FILEPATH,
        monitor="val_loss",
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode="min",
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [ ]:
'''
uncomment this for when you have the baseline model trained
'''
# with strategy.scope():
#     model = keras.models.load_model("/kaggle/input/models/jkong05/<NEW-10CLS-BASELINE-SLUG>/best.model.keras")

In [ ]:
H = model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)

Epoch 1/20
5126/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640

2026-04-28 19:24:33.099991: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.237965: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.553611: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.695667: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:34.509188: E external/local_xla/xla/stream_

5127/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640

2026-04-28 19:25:51.217259: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.359820: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.709253: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.851758: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:52.634989: E external/local_xla/xla/stream_


Epoch 1: val_loss improved from inf to 2.98957, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 511s 96ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640 - val_accuracy: 0.1196 - val_loss: 2.9896 - val_top_3_accuracy: 0.2792
Epoch 2/20
5125/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1179 - loss: 3.0069 - top_3_accuracy: 0.2762
Epoch 2: val_loss improved from 2.98957 to 2.98634, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 191s 37ms/step - accuracy: 0.1179 - loss: 3.0069 - top_3_accuracy: 0.2762 - val_accuracy: 0.1269 - val_loss: 2.9863 - val_top_3_accuracy: 0.2858
Epoch 3/20
5125/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1195 - loss: 3.0032 - top_3_accuracy: 0.2787
Epoch 3: val_loss improved from 2.98634 to 2.98589, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 190s 37ms/step - accuracy: 0.1195 - loss: 3.0032 -

<h3> Visualization <h3>

In [ ]:
def create_accuracy_loss_plot(H):
    acc = H.history['accuracy']
    val_acc = H.history['val_accuracy']
    loss = H.history['loss']
    val_loss = H.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, acc, label='Train')
    ax1.plot(epochs, val_acc, label='Val')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(epochs, loss, label='Train')
    ax2.plot(epochs, val_loss, label='Val')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.tight_layout()
    fig.savefig('/kaggle/working/accuracy_loss.png', bbox_inches='tight', dpi=150)
    plt.show()

def create_confusion_matrix(model, test_data, classes):
    preds = model.predict(test_data).argmax(axis=1)
    true_labels = np.concatenate([y for _, y in test_data], axis=0)

    cm = confusion_matrix(true_labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)

    fig, ax = plt.subplots(figsize=(16, 16))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    plt.tight_layout()
    fig.savefig('/kaggle/working/confusion_matrix.png', bbox_inches='tight', dpi=150)
    plt.show()
    

def create_heatmap(img_array, model):
    base = model.get_layer('efficientnetb0')

    inner_grad_model = tf.keras.Model(
        inputs=base.inputs,
        outputs=[base.get_layer('top_activation').output, base.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, base_out = inner_grad_model(img_array, training=False)
        tape.watch(conv_outputs)

        x = model.get_layer('global_average_pooling2d')(base_out)
        x = model.get_layer('dropout')(x, training=False)
        predictions = model.get_layer('dense')(x)

        pred_idx = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_idx]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_idx.numpy()


def display_heatmap(img_path, true_idx, model, classes):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img_array = tf.expand_dims(tf.cast(img, tf.float32), 0)

    heatmap, pred_idx = create_heatmap(img_array, model)

    heatmap = heatmap.astype(np.float32)
    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    orig = np.uint8(img.numpy())
    overlay = cv2.addWeighted(orig, 0.6, heatmap, 0.4, 0)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
    ax1.imshow(orig)
    ax1.set_title('Original') 
    ax1.axis('off')
    ax2.imshow(heatmap) 
    ax2.set_title('Heatmap') 
    ax2.axis('off')
    ax3.imshow(overlay)
    ax3.set_title(f'Pred: {classes[pred_idx]}\nTrue: {classes[true_idx]}')
    ax3.axis('off')
    plt.tight_layout()
    fig.savefig(f'/kaggle/working/heatmap_{os.path.basename(img_path)}.png', bbox_inches='tight', dpi=150)
    plt.show()

def top_k_predictions_df(model, df, classes, k=3, batch_size=64):
    """Run model over df['filepath'] and return a DataFrame with top-k labels + probabilities per row."""
    paths = df['filepath'].values
    ds = tf.data.Dataset.from_tensor_slices(paths)
    def _load(p):
        img = tf.io.read_file(p)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, [224, 224])
        return tf.cast(img, tf.float32)
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    probs = model.predict(ds, verbose=1)  # shape: (N, num_classes)
    top_idx = np.argsort(-probs, axis=1)[:, :k]              # (N, k)
    top_probs = np.take_along_axis(probs, top_idx, axis=1)   # (N, k)

    out = pd.DataFrame({'filepath': paths})
    if 'sector' in df.columns:
        out['true_label'] = df['sector'].values
    for i in range(k):
        out[f'top{i+1}_label'] = [classes[j] for j in top_idx[:, i]]
        out[f'top{i+1}_prob']  = top_probs[:, i]
    return out

def plot_top_k_results(top_k_df, k=3):
    label_cols = [f'top{i+1}_label' for i in range(k)]
    in_top1 = top_k_df['top1_label'] == top_k_df['true_label']
    in_topk = np.any([top_k_df[c] == top_k_df['true_label'] for c in label_cols], axis=0)

    df = top_k_df.assign(in_top1=in_top1, in_topk=in_topk)
    per_class = df.groupby('true_label').agg(
        top1=('in_top1', 'mean'),
        topk=('in_topk', 'mean'),
        n=('in_top1', 'size'),
    ).sort_values('topk')

    overall_top1 = in_top1.mean()
    overall_topk = in_topk.mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

    y = np.arange(len(per_class))
    ax1.barh(y, per_class['topk'] * 100, color='steelblue', label=f'Top-{k}')
    ax1.barh(y, per_class['top1'] * 100, color='darkorange', label='Top-1')
    ax1.set_yticks(y)
    ax1.set_yticklabels(per_class.index, fontsize=9)
    ax1.set_xlabel('Accuracy (%)')
    ax1.set_title(f'Per-class accuracy  (overall top-1={overall_top1:.1%}, top-{k}={overall_topk:.1%})')
    ax1.axvline(overall_topk * 100, color='steelblue', linestyle='--', alpha=0.4)
    ax1.axvline(overall_top1 * 100, color='darkorange', linestyle='--', alpha=0.4)
    ax1.legend(loc='lower right')
    ax1.set_xlim(0, 100)

    correct_probs = top_k_df.loc[in_top1, 'top1_prob']
    wrong_probs = top_k_df.loc[~in_top1, 'top1_prob']
    bins = np.linspace(0, 1, 40)
    ax2.hist(wrong_probs, bins=bins, alpha=0.6, label='Top-1 wrong', color='crimson')
    ax2.hist(correct_probs, bins=bins, alpha=0.6, label='Top-1 correct', color='seagreen')
    ax2.set_xlabel('Top-1 probability')
    ax2.set_ylabel('Count')
    ax2.set_title('Confidence distribution by correctness')
    ax2.legend()

    plt.tight_layout()
    fig.savefig('/kaggle/working/top_k_results.png', bbox_inches='tight', dpi=150)
    plt.show()

    return per_class


In [ ]:
create_accuracy_loss_plot(H)

In [ ]:
create_confusion_matrix(model, test_data, classes)

In [ ]:
for img_path, true_idx in zip(test_df['filepath'].iloc[:5], test_df['label_idx'].iloc[:5]):
    display_heatmap(img_path, true_idx, model, classes)

In [ ]:
# # tuning
# base = model.get_layer('efficientnetb0')

# unfreeze_blocks = ('block7', 'top')

# for layer in base.layers:
#     if layer.name.startswith(unfreeze_blocks) and not isinstance(layer, layers.BatchNormalization):
#         layer.trainable = True
#     else:
#         layer.trainable = False

# with strategy.scope():
#     model.compile(
#         optimizer=keras.optimizers.Adam(learning_rate=2e-5, clipnorm=1.0),
#         loss='sparse_categorical_crossentropy',
#         metrics=[
#             'accuracy',
#             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_SparseTopKCategoricalAccuracy'),
#         ]
#     )

# tuned_callbacks = [
#     keras.callbacks.ModelCheckpoint(
#         TUNED_CHECKPOINT_FILEPATH,
#         monitor="val_loss",
#         verbose=1,
#         save_best_only=True,
#         save_weights_only=False,
#         mode="min",
#     ),
#     keras.callbacks.EarlyStopping(
#         monitor="val_loss",
#         patience=5,
#         restore_best_weights=True,
#         verbose=1,
#     ),
#     keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
#     keras.callbacks.TerminateOnNaN()
# ]

In [ ]:
# H2 = model.fit(train_data, validation_data=val_data, epochs=50, callbacks=tuned_callbacks)

In [ ]:
# create_accuracy_loss_plot(H2)

In [ ]:
# create_confusion_matrix(model, test_data, classes)

In [ ]:
# for img_path, true_idx in zip(test_df['filepath'].iloc[:5], test_df['label_idx'].iloc[:5]):
#     display_heatmap(img_path, true_idx, model, classes)

In [ ]:
# test_top3 = top_k_predictions_df(model, test_df, classes, k=3)
# test_top3.to_csv('/kaggle/working/test_top3_predictions.csv', index=False)
# test_top3.head(10)

In [ ]:
# per_class = plot_top_k_results(test_top3, k=3)
# per_class